# Intrinsic and extrinsic scalar curvature by grid resolution

This notebook uses the two 2D ellipsoids and the 2D hyperboloid in `test_curvature.py` (the non-slow analytic cases). Each case has ten grid sizes, ending at its corresponding pointwise test size. The methods receive exactly the same sampled embedding, coordinate spacings, periodic axis, patches, and device. We compare against the analytic scalar curvature, against each other, and by elapsed time. These are finite-difference convergence diagnostics, not independent proof from method agreement.

The non-slow total-curvature test also uses the same two ellipsoids at 10,223 × 10,344. Set `INCLUDE_TOTAL_TEST_GRID = True` to add that grid schedule. Its largest arrays need multiple gigabytes and can take a long time. The notebook starts without saved outputs.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import torch

from microscope.computations_grid.curvature import scalar_curvature as intrinsic_curvature
from microscope.computations_grid.curvature_extrinsic import scalar_curvature as extrinsic_curvature
from microscope.cyclic_dimensions import get_difference_intervals
from microscope.manifold_examples.ellipsoid import ellipsoid_scalar_curvature
from microscope.manifold_examples.hyperboloid import hyperboloid_scalar_curvature
from microscope.manifold_examples.sampling_grid import sample_ellipsoid_on_grid, sample_hyperboloid_on_grid

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
INCLUDE_TOTAL_TEST_GRID = False
N_SIZES = 10

In [ ]:
# Parameters are copied from the non-slow cases in test_curvature.py.
CASE_SPECS = {
    "sphere / regular": dict(kind="ellipsoid", axes=[2, 2, 2], maximum=(1006, 1000), patches=[56, 56], fraction=.8),
    "ellipsoid / regular": dict(kind="ellipsoid", axes=[1, 1.5, 3], maximum=(1006, 1000), patches=[56, 56], fraction=.8),
    "sphere / irregular": dict(kind="ellipsoid", axes=[2, 2, 2], maximum=(1223, 1344), patches=[34, 42], fraction=.8),
    "ellipsoid / irregular": dict(kind="ellipsoid", axes=[1, 1.5, 3], maximum=(1223, 1344), patches=[34, 42], fraction=.8),
    "hyperboloid": dict(kind="hyperboloid", axes=[5, 1, 3], maximum=(1006, 1000), patches=[56, 56]),
}
if INCLUDE_TOTAL_TEST_GRID:
    for name, axes in (("sphere / total-test grid", [2, 2, 2]),
                       ("ellipsoid / total-test grid", [1, 1.5, 3])):
        CASE_SPECS[name] = dict(kind="ellipsoid", axes=axes, maximum=(10223, 10344),
                                patches=[1034, 1042], fraction=.9999)


def grid_sizes(maximum, count=N_SIZES):
    sizes = np.rint(np.geomspace(16, np.asarray(maximum), count)).astype(int)
    sizes[-1] = maximum
    assert len({tuple(pair) for pair in sizes}) == count
    return [tuple(pair) for pair in sizes]


def sample_case(spec, sizes):
    cyclic_dimensions = [1]
    if spec["kind"] == "ellipsoid":
        points, axes, phis, _ = sample_ellipsoid_on_grid(
            semi_axes=spec["axes"], n_samples_per_dim=list(sizes), ambient_space_dim=3,
            apply_random_isometry=False, fraction_of_angles=spec["fraction"]
        )
        range_sizes = [(2*spec["fraction"] - 1)*np.pi, 2*np.pi]
        truth = ellipsoid_scalar_curvature(axes, phis)[3:-3, :]
    else:
        points, axes, phis, _ = sample_hyperboloid_on_grid(
            semi_axes=spec["axes"], n_samples_per_dim=list(sizes), ambient_space_dim=3,
            apply_random_isometry=False
        )
        range_sizes = [2., 2*np.pi]
        truth = hyperboloid_scalar_curvature(axes, phis)[3:-3, :]
    intervals = get_difference_intervals(list(sizes), range_sizes, cyclic_dimensions)
    return points, truth, intervals, cyclic_dimensions

In [ ]:
def compare_case_at_size(name, spec, sizes, device=DEVICE):
    points, truth, intervals, cyclic_dimensions = sample_case(spec, sizes)
    kwargs = dict(difference_intervals=intervals, cyclic_dimensions=cyclic_dimensions,
                  patch_sizes=spec["patches"], device=device)
    row = dict(case=name, shape=sizes, points=int(np.prod(sizes)))
    predictions = {}
    for label, method in (("intrinsic", intrinsic_curvature),
                          ("extrinsic", extrinsic_curvature)):
        if str(device).startswith("cuda"):
            torch.cuda.synchronize(device)
        start = perf_counter()
        predictions[label] = method(points, **kwargs)
        if str(device).startswith("cuda"):
            torch.cuda.synchronize(device)
        row[label + "_seconds"] = perf_counter() - start
        assert predictions[label].shape == truth.shape
        error = np.abs(predictions[label] - truth)
        row[label + "_mae"] = float(error.mean())
        row[label + "_max_error"] = float(error.max())
    row["method_mae"] = float(np.abs(predictions["intrinsic"] - predictions["extrinsic"]).mean())
    return row


# Change CASE_SPECS or N_SIZES above to limit a local run.
results = []
for name, spec in CASE_SPECS.items():
    for sizes in grid_sizes(spec["maximum"]):
        row = compare_case_at_size(name, spec, sizes)
        results.append(row)
        print(f"{name:27s} {sizes!s:15s}  intrinsic {row['intrinsic_mae']:.3g}  "
              f"extrinsic {row['extrinsic_mae']:.3g}  difference {row['method_mae']:.3g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for name in CASE_SPECS:
    rows = [row for row in results if row["case"] == name]
    if not rows:
        continue
    x = [row["points"] for row in rows]
    axes[0].loglog(x, [row["intrinsic_mae"] for row in rows], "o-", label=f"{name}: intrinsic")
    axes[0].loglog(x, [row["extrinsic_mae"] for row in rows], "s--", label=f"{name}: extrinsic")
    axes[0].loglog(x, [row["method_mae"] for row in rows], ":", label=f"{name}: difference")
    axes[1].loglog(x, [row["intrinsic_seconds"] for row in rows], "o-", label=f"{name}: intrinsic")
    axes[1].loglog(x, [row["extrinsic_seconds"] for row in rows], "s--", label=f"{name}: extrinsic")
axes[0].set(xlabel="grid points", ylabel="mean absolute scalar-curvature error",
            title="Analytic error and method agreement")
axes[1].set(xlabel="grid points", ylabel="elapsed seconds", title="Runtime, including patches")
for ax in axes:
    ax.grid(True, which="both", alpha=.25)
    ax.legend(fontsize=7, ncol=2)
plt.show()

Check the analytic error curves before interpreting method agreement. The two methods use the same embedding and grid, so their difference can shrink even if both retain a shared sampling bias. The two optional total-test grids approach a coordinate singularity; inspect the maximum errors and conditioning there as well as the mean.